In [28]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

In [29]:
import re
import torch
import numpy as np
import torch.nn as nn
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset



from sklearn.model_selection import train_test_split
from models.lstm_model import LSTM_Model
from utils import EarlyStopping, get_device,EpochTrainer
from datasets import load_dataset

import pandas as pd

# Parameters

In [30]:
model_output_filename = "../../checkpoints/sentiment_checkpoint.pt"
IsDownload_data = False

data_src_path = "zeroshot/twitter-financial-news-sentiment"
data_save_path = '../../data/phrasebank_local.csv'

## model config

In [31]:
input_size = None
batch_size = 32
epochs = 100
hidden_size = 128
num_output = 3
num_layers = 1
dropout = 0.2
bidirectional = True
patience = 5
test_size = 0.3
lr = 0.008

PAD_IDX = 0
UNK_IDX = 1
embedding_dim = 200
max_seq_len = 60


In [32]:
eval_method = 'Accuracy'

# Load and prepare training data

## Download dataset and save as csv

In [33]:
if IsDownload_data:

    dataset = load_dataset(data_src_path, split="train")
    
    df = pd.DataFrame({
        'text': [row['text'] for row in dataset],
        'label': [row['label'] for row in dataset]
    })
    
    df.to_csv( data_save_path , index=False)

## Load Downloaded dataset from csv

In [34]:
df = pd.read_csv(data_save_path)

## Preprocess dataset

Goal: Convert raw text sentences into fixed-length numeric sequences for the model
Overview:
1. Tokenize : Convert sentence to lowercase, remove punctuation, split into words
2. Build Vocab : Count word frequency across entire dataset, keep words appearing >=2 times,
   assign each word a unique ID (reserve 0 for PAD, 1 for UNK)
3. Encode : Replace each word with its vocab ID (unknown words → UNK_IDX)
4. Pad/Truncate : Ensure all sequences are length 40 (pad short ones, truncate long ones)
       
Result: All sentences become fixed-length arrays of integers, ready for embedding layer
Example flow:
- Raw text:        "Strong earnings! Q3 beat."
- After tokenize:  ["strong", "earnings", "q3", "beat"]
- Vocab mapping:   {"<PAD>": 0, "<UNK>": 1, "strong": 2, "earnings": 3, "beat": 4, ...}
- After encode:    [2, 3, 1, 4]                    (q3 not in vocab → use UNK_IDX=1)
- After pad (40):  [2, 3, 1, 4, 0, 0, 0, ..., 0]   (pad to 40 with zeros)

In [35]:
texts = df['text'].tolist()
labels = df['label'].tolist()
label_map = {"negative": 0, "neutral": 1, "positive": 2}

### Split data into training and testing

In [36]:
texts_train, texts_test, labels_train, labels_test = train_test_split(
    texts,
    labels,
    test_size=test_size,
    random_state=42,
    stratify=labels,
)

### Function to Tokenize sentense

In [37]:
def tokenize(text):
    text = re.sub(r"[^a-zA-Z0-9 ]+", " ", str(text).lower())
    return text.split()

### Build vocabulary

In [38]:
counter = Counter()

for text in texts_train:
    counter.update(tokenize(text))

In [39]:
vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}

for token, freq in counter.items():
    if freq >= 2:
        vocab[token] = len(vocab)

print(f"Vocab size: {len(vocab)}")

Vocab size: 5984


### Convert to sequences

In [40]:
def encode_text(text, max_len = max_seq_len):
    ids = [vocab.get(tok, UNK_IDX) for tok in tokenize(text)]
    ids = ids[:max_len]
    ids += [PAD_IDX] * (max_len - len(ids))
    
    return ids

In [41]:
X_train = np.array([encode_text(text) for text in texts_train], dtype=np.int64)
y_train = np.array(labels_train, dtype=np.int64)

X_test = np.array([encode_text(text) for text in texts_test], dtype=np.int64)
y_test = np.array(labels_test, dtype=np.int64)

In [42]:
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Prepare Model

## Prepare config of the model

In [43]:
preprocess_config = {
    "seq_length": max_seq_len,
    "padding_idx": PAD_IDX,
    "unk_idx": UNK_IDX,
    "task_type": "classification",
}

In [44]:
model_config = {
    "input_size": input_size,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_layers": num_layers,
    "dropout": dropout,
    "bidirectional": bidirectional,
    "vocab_size": len(vocab),
    "embedding_dim": embedding_dim,
    "padding_idx": PAD_IDX,
}

## Get Model

In [45]:
device = get_device()

In [46]:
model = LSTM_Model(**model_config).to(device)

In [47]:
optimizer = torch.optim.Adam(model.parameters(),  lr = lr)

In [48]:
class_counts = np.bincount(y_train)  # [count_class0, count_class1, count_class2]
class_weights = len(y_train) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

## Early stopping and checkpoint setup

In [49]:
early_stopping = EarlyStopping(
    patience = patience,
    path = model_output_filename,
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "vocab": vocab,
        "label_map": label_map,
        "max_len": preprocess_config["seq_length"],
    },
)

# Loop epochs and train model

In [50]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = eval_method
)

In [51]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():        
        
        print(f"Epoch {epoch} | Train Loss: {avg_train_loss / len(train_loader):.4f} | Val Loss: {avg_val_loss / len(test_loader):.4f} | {key}: {value:.4f}")
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break
        

Epoch 0 | Train Loss: 0.0040 | Val Loss: 0.0079 | Accuracy: 0.7468
Epoch 1 | Train Loss: 0.0018 | Val Loss: 0.0086 | Accuracy: 0.7489
Epoch 2 | Train Loss: 0.0007 | Val Loss: 0.0118 | Accuracy: 0.7813
Epoch 3 | Train Loss: 0.0003 | Val Loss: 0.0160 | Accuracy: 0.7848
Epoch 4 | Train Loss: 0.0001 | Val Loss: 0.0185 | Accuracy: 0.7800
Epoch 5 | Train Loss: 0.0000 | Val Loss: 0.0190 | Accuracy: 0.7880
Early stopping triggered! Training stopped.
